<a href="https://colab.research.google.com/github/Locket51/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/Locket51/flyrank-ml-internship.git
%cd flyrank-ml-internship

Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 124, done.
remote: Counting objects: 100% (124/124), done.
remote: Compressing objects: 100% (80/80), done.
remote: Total 124 (delta 34), reused 97 (delta 28), pack-reused 0 (from 0)
Receiving objects: 100% (124/124), 1.85 MiB | 14.32 MiB/s, done.
Resolving deltas: 100% (34/34), done.
/content/flyrank-ml-internship


In [3]:
!pip install -q duckdb huggingface_hub

from google.colab import userdata
import duckdb, os

os.environ["HF_TOKEN"] = userdata.get("HF")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")

print(con.execute("""
    SELECT * FROM read_parquet(
      'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet'
    ) LIMIT 3
""").df())

  report_date           client_hash_id           content_hash_id  \
0  2025-01-27  client_9958f0a7ae1df715  content_3b70a18ea133b2bb   
1  2025-01-27  client_9958f0a7ae1df715  content_fe8e8155ce1d47a2   
2  2025-01-27  client_9958f0a7ae1df715  content_b4462a1b90640058   

   client_has_gsc  client_has_ga4  gsc_data_available  ga4_data_available  \
0            True            True                True               False   
1            True            True                True               False   
2            True            True                True               False   

   gsc_impressions  gsc_clicks  gsc_sum_position  ...  sessions_ai  \
0               30           0               115  ...            0   
1                5           0               358  ...            0   
2                1           0                34  ...            0   

   ai_chatgpt  ai_perplexity  ai_gemini  ai_copilot  ai_claude  ai_meta  \
0           0              0          0           0          0

In [4]:
!pip install -q duckdb huggingface_hub

from google.colab import userdata
import duckdb, os

os.environ["HF_TOKEN"] = userdata.get("HF")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")
print("connection ready")

connection ready


In [5]:
print(con.execute("""
  SELECT
    COUNT(*) AS rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS contents,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date
  FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
  WHERE month = '2026-03'
""").df())

print(con.execute("""
  SELECT
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS with_gsc,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS with_ga4,
    COUNT(*) AS total
  FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
  WHERE month = '2026-03'
""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      rows  clients  contents   min_date   max_date
0  9841378       55    331437 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    with_gsc  with_ga4    total
0  3611061.0  413966.0  9841378


## 1–2. My data contract, in plain words

**One row means:** one content item, on one client, on one calendar day. The grain of `fact_content_daily_performance` is `report_date × client_hash_id × content_hash_id`. Every decision I make from this table has to respect that grain.

**Tables I will use:**
- `fact_content_daily_performance` — the daily fact table (the primary source).
- `dim_content` — content metadata, joined on `content_hash_id`, if I need a static feature like `content_age_days` or `content_type`.
- `dim_clients` — client-level history start dates (`gsc_data_start`, `ga4_data_start`), joined on `client_hash_id`, only when I need to check whether a client was even being tracked yet.

**Time window:** a mid-panel month, `month = '2026-03'`. I am deliberately not using `2026-06` (the `_sample` table) because that is the freshest month and would be the natural outcome window of a past→future label. Iterating there would mean peeking at the answer.

**What I predict or rank:** a binary proxy label `is_declining` for each content item within the month — "did this page's daily impressions in the second half of March drop meaningfully versus the first half, controlling for volume?" This is a **within-month proxy**, not the capstone's future-window label. It is deliberately easier so the loop runs end to end. The capstone upgrades the same shape (features from an earlier window → outcome in a later window) to the full warehouse.

**What I deliberately exclude:** any column that is computed *from* the outcome window — anything derived from future days, and anything that encodes the trend itself (`trend_pct`, `trend_direction`, current-period buckets). Also excluded by policy: `client_hash_id` and `content_hash_id` as model features (they are keys, not signals), and any raw query/URL/title field (none exist in the release anyway).

In [6]:
!pip install -q duckdb huggingface_hub

from google.colab import userdata
import duckdb, os

os.environ["HF_TOKEN"] = userdata.get("HF")

con = duckdb.connect()
con.execute("INSTALL httpfs; LOAD httpfs;")
con.execute(f"CREATE SECRET hf (TYPE HUGGINGFACE, TOKEN '{os.environ['HF_TOKEN']}');")
print("connection ready")

connection ready


## 1–2. My data contract, in plain words

**One row means:** one content item, on one client, on one calendar day. The grain of `fact_content_daily_performance` is `report_date × client_hash_id × content_hash_id`. Every decision I make from this table has to respect that grain.

**Tables I will use:**
- `fact_content_daily_performance` — the daily fact table (the primary source).
- `dim_content` — content metadata, joined on `content_hash_id`, if I need a static feature like `content_age_days` or `content_type`.
- `dim_clients` — client-level history start dates (`gsc_data_start`, `ga4_data_start`), joined on `client_hash_id`, only when I need to check whether a client was even being tracked yet.

**Time window:** a mid-panel month, `month = '2026-03'`. I am deliberately not using `2026-06` (the `_sample` table) because that is the freshest month and would be the natural outcome window of a past→future label. Iterating there would mean peeking at the answer.

**What I predict or rank:** a binary proxy label `is_declining` for each content item within the month — "did this page's daily impressions in the second half of March drop meaningfully versus the first half, controlling for volume?" This is a **within-month proxy**, not the capstone's future-window label. It is deliberately easier so the loop runs end to end. The capstone upgrades the same shape (features from an earlier window → outcome in a later window) to the full warehouse.

**What I deliberately exclude:** any column that is computed *from* the outcome window — anything derived from future days, and anything that encodes the trend itself (`trend_pct`, `trend_direction`, current-period buckets). Also excluded by policy: `client_hash_id` and `content_hash_id` as model features (they are keys, not signals), and any raw query/URL/title field (none exist in the release anyway).

## 3. Verification queries (three, on `month = '2026-03'`)

Three small queries to prove the contract: that the grain is what I said, what the row count and date span actually are, and how many rows survive an `IS TRUE` availability filter.

In [7]:
# Query 1 — is one row really one (report_date x client x content)?
q1 = con.execute("""
  SELECT
    COUNT(*)                                                AS rows,
    COUNT(DISTINCT report_date || '|' || client_hash_id || '|' || content_hash_id) AS distinct_keys
  FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
  WHERE month = '2026-03'
""").df()
q1["grain_is_1to1"] = q1["rows"] == q1["distinct_keys"]
q1

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,rows,distinct_keys,grain_is_1to1
0,9841378,9841378,True


In [8]:
# Query 2 — row count and date span for the mid-panel month
q2 = con.execute("""
  SELECT
    COUNT(*)                       AS rows,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS contents,
    MIN(report_date)               AS min_date,
    MAX(report_date)               AS max_date
  FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
  WHERE month = '2026-03'
""").df()
q2

,rows,clients,contents,min_date,max_date
0,9841378,55,331437,2026-03-01,2026-03-31


In [9]:
# Query 3 — how many rows survive an IS TRUE availability filter
q3 = con.execute("""
  SELECT
    COUNT(*)                                                          AS total,
    SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END)       AS with_gsc,
    SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END)       AS with_ga4,
    SUM(CASE WHEN gsc_data_available IS TRUE
              AND ga4_data_available IS TRUE THEN 1 ELSE 0 END)       AS with_both
  FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
  WHERE month = '2026-03'
""").df()
q3["pct_gsc"] = (q3["with_gsc"] / q3["total"]).round(4)
q3["pct_ga4"] = (q3["with_ga4"] / q3["total"]).round(4)
q3

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total,with_gsc,with_ga4,with_both,pct_gsc,pct_ga4
0,9841378,3611061.0,413966.0,364347.0,0.3669,0.0421


### Five features (max), each with an "available when?" line

Five features, all knowable from days that fall **before** the second half of March. Each line justifies the timing.

1. **`impressions_first_half`** — total GSC impressions in 2026-03-01 … 2026-03-15. *Available because* it comes from days strictly before the label window.
2. **`clicks_first_half`** — total GSC clicks over the same 15 days. *Available because* it uses only prior-window events.
3. **`avg_position_first_half`** — mean `gsc_sum_position` over the same window. *Available because* position is observed on the same days, not after.
4. **`ctr_first_half`** — `clicks_first_half / impressions_first_half`. *Available because* derived only from prior-window measurements.
5. **`days_with_impressions_first_half`** — count of distinct days with any impressions in the first half. *Available because* it is a count over the prior window.

For the label: **`is_declining`** = did `impressions_second_half` fall meaningfully versus `impressions_first_half`? Built from the second half of March, which is the *target* window and never used as a feature.

In [10]:
# Build the five-feature frame + within-month proxy label, on month = '2026-03'
features = con.execute("""
  WITH daily AS (
    SELECT
      content_hash_id,
      client_hash_id,
      report_date,
      gsc_impressions,
      gsc_clicks,
      gsc_sum_position,
      CASE WHEN report_date <= DATE '2026-03-15' THEN 'first' ELSE 'second' END AS half
    FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/*/*.parquet')
    WHERE month = '2026-03'
      AND gsc_data_available IS TRUE
  ),
  agg AS (
    SELECT
      content_hash_id,
      client_hash_id,
      SUM(CASE WHEN half = 'first' THEN gsc_impressions ELSE 0 END)            AS impressions_first_half,
      SUM(CASE WHEN half = 'first' THEN gsc_clicks ELSE 0 END)                 AS clicks_first_half,
      AVG(CASE WHEN half = 'first' THEN gsc_sum_position END)                  AS avg_position_first_half,
      COUNT(DISTINCT CASE WHEN half = 'first' AND gsc_impressions > 0
                          THEN report_date END)                                AS days_with_impressions_first_half,
      SUM(CASE WHEN half = 'second' THEN gsc_impressions ELSE 0 END)           AS impressions_second_half
    FROM daily
    GROUP BY content_hash_id, client_hash_id
  )
  SELECT
    content_hash_id,
    client_hash_id,
    impressions_first_half,
    clicks_first_half,
    avg_position_first_half,
    days_with_impressions_first_half,
    impressions_second_half,
    CASE
      WHEN impressions_first_half >= 50
        THEN CASE WHEN impressions_second_half < 0.7 * impressions_first_half THEN 1 ELSE 0 END
      ELSE NULL
    END AS is_declining,
    impressions_first_half * 1.0 / NULLIF(days_with_impressions_first_half, 0) AS avg_daily_impressions_first_half
  FROM agg
  WHERE impressions_first_half >= 50
""").df()

features["ctr_first_half"] = (
    features["clicks_first_half"] / features["impressions_first_half"]
)

print("frame shape:", features.shape)
print("\nlabel distribution:")
print(features["is_declining"].value_counts(dropna=False))
print("\nhead:")
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

frame shape: (92548, 10)

label distribution:
is_declining
0    72410
1    20138
Name: count, dtype: int64

head:


,content_hash_id,client_hash_id,impressions_first_half,clicks_first_half,avg_position_first_half,days_with_impressions_first_half,impressions_second_half,is_declining,avg_daily_impressions_first_half,ctr_first_half
0,content_7a105f548d9c6916,client_73cda7b4e4f265ea,4173.0,6.0,1742.933333,15,2350.0,1,278.200000,0.001438
1,content_905aa32a0230694e,client_73cda7b4e4f265ea,89.0,0.0,20.600000,15,60.0,1,5.933333,0.000000
2,content_a3ea9792f793ec72,client_73cda7b4e4f265ea,245.0,0.0,66.733333,15,208.0,0,16.333333,0.000000
3,content_36c36abc7650d7af,client_73cda7b4e4f265ea,3705.0,3.0,1555.533333,15,1925.0,1,247.000000,0.000810
4,content_05434271b257bb68,client_73cda7b4e4f265ea,628.0,1.0,220.466667,15,793.0,0,41.866667,0.001592


### 4. The trap — a deliberate leak, and its removal

The assignment asks me to spring the leakage trap on purpose: add ONE label-derived column, watch the score jump toward perfect, then delete it and keep the honest number. This is the leakage lesson from notebook 02, performed on real warehouse data.

The rule I am following: **the leak is a diagnostic, not a feature.** It goes in, I measure, it comes out, the honest model stays.

In [11]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

honest_features = [
    "impressions_first_half",
    "clicks_first_half",
    "avg_position_first_half",
    "ctr_first_half",
    "days_with_impressions_first_half",
]

model_df = features.dropna(subset=["is_declining"]).copy()
y = model_df["is_declining"].astype(int)

def quick_auc(cols, data):
    X = data[cols].fillna(0).values
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
    return cross_val_score(clf, X, y, cv=3, scoring="roc_auc").mean()

honest_auc = quick_auc(honest_features, model_df)
print(f"HONEST AUC (5 real features): {honest_auc:.3f}")

# --- deliberate leak ---
model_df["LEAK_second_half_impressions"] = model_df["impressions_second_half"]
leaked_auc = quick_auc(honest_features + ["LEAK_second_half_impressions"], model_df)
print(f"LEAKED AUC (with second-half impressions added): {leaked_auc:.3f}")

# --- remove the leak, keep the honest number ---
del model_df["LEAK_second_half_impressions"]
honest_auc_again = quick_auc(honest_features, model_df)
print(f"HONEST AUC after removing the leak: {honest_auc_again:.3f}")

HONEST AUC (5 real features): 0.611
LEAKED AUC (with second-half impressions added): 1.000
HONEST AUC after removing the leak: 0.611


## 4. One named limitation of my slice

**The slice is history-imbalanced across clients.** In `2026-03` there are 55 distinct `client_hash_id` values and 331,437 distinct `content_hash_id` values across 9,841,378 rows. Only **4.2%** of those rows have `ga4_data_available IS TRUE`, meaning almost all rows in this month are search data only. Any feature that depends on GA4 (sessions, engagement, scroll) will silently shrink the training set to a small, non-random subset of clients. I state this plainly, and I only build GA4-based features when the model is trained on the GA4-available slice alone.

## 5. Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under work/notebooks/ — then submit your repo URL on the card